# A first look at a single-compartment neuron with Hodgkin & Huxley conductances

Stay alert! There's some **potential** for **action** here!

This notebook grades your answers for you, and **each student gets a slightly
different neuron** -- so the firing thresholds you find at the end are your own.

## Step 1: Setup

In [ ]:
# Setup inline plotting
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# For Google Colab, this line installs NEURON
#!pip install neuron quantities

In [ ]:
# We will let this library handle unit conversion for us
import quantities as pq
from quantities import um, nS, mV, cm, ms, nA, S, uF, Hz, degrees, s

In [ ]:
# Import and initialize NEURON
import neuron
from neuron import h
h.load_file("stdrun.hoc")

In [ ]:
# Import other modules we need
import numpy as np

# Progress bars: the simulations below take a while, and a bar is much nicer than
# staring at a cell that gives no sign of life.
from tqdm.notebook import trange, tqdm

## Step 1b: Load your personal exercise parameters

This notebook will grade your answers for you, and **each student gets a slightly
different neuron**. The cell below fetches your own soma length, leak conductance
and a starting current to inject; the rest of the notebook builds the cell from
them, so the firing thresholds you submit at the end are specific to the neuron
you simulated.

How much current a neuron needs before it will fire depends on its size and how
leaky it is, so **there is no single current that suits every one of us** -- hence
a starting current of your own, chosen to be just enough to get your cell going.

`grading.load()` reads the launch file the platform writes next to this notebook.
You never handle any tokens or URLs yourself.

In [ ]:
from obi_notebook import grading

assignment = grading.load()

# Your personal parameters for this assignment.
soma_length = assignment.params["soma_length_um"]
g_leak = assignment.params["g_leak_nS"]
initial_current = assignment.params["initial_current_nA"]

print(f"Your soma length:       {soma_length} um")
print(f"Your leak conductance:  {g_leak} nS")
print(f"Your starting current:  {initial_current} nA")
print(f"Exercises to submit:    {assignment.exercise_keys}")

## Step 2: Define the circuit
We will use a single compartment, called a "Section" (more on that in next lectures). <br>
It has a cylindrical geometry with length "L" and a diameter "diam", and a specific capacitance "cm" (capacitance per area) <br>
**Unit conversion is a common source of error, so we will be explicit with our units.** 

In [ ]:
soma = h.Section()

### Query NEURON for the expected units for soma.L & soma.diam

In [ ]:
[h.units(x) for x in ["L", "diam"]]

In [ ]:
# soma.L is YOUR personal value, loaded in Step 1b above.
soma.L = soma_length * um
soma.diam =  10 * um

In [ ]:
volume = soma(0.5).volume() * um**3

In [ ]:
area = soma(0.5).area() * um**2

In [ ]:
area

In [ ]:
volume

### Assign the membrane capacitance "everywhere"

In [ ]:
h.units("cm")  # Query the expected units

In [ ]:
specific_membrane_capacitance = 1 * uF/cm**2

In [ ]:
for sec in soma.wholetree():
    sec.cm = specific_membrane_capacitance #  specific membrane capacitance (micro Farads / cm^2)
    sec.Ra = 100

### Add the Hodgkin-Huxley conductances

In [ ]:
# This model includes the transient Na+, persistent K+ and the leak conductances
soma.insert("hh")

That's almost too easy!

### Parametize the leak conductance G = 1/R

In [ ]:
G = g_leak * nS  # R = 1/G in our RC circuit -- YOUR personal value, see Step 1b

In [ ]:
# The voltage we START the simulation at. This is NOT the resting potential:
# with HH channels the cell settles wherever its own currents balance. Look at
# the trace before the stimulus begins to see where that is.
v_init = -70*mV

In [ ]:
tau_m = (specific_membrane_capacitance * area / G).rescale(ms)

In [ ]:
tau_m

In [ ]:
# Assign the leak conductance everywhere
for seg in soma:
    seg.hh.gl = (G/area).rescale(S/cm**2)  # Compute specific conductance, and rescale to units of 'S/cm2'
    seg.hh.el = -54.3

### Inspect our parameters

In [ ]:
soma.psection()

In [ ]:
soma.nseg

### Add a current injection

In [ ]:
stim = h.IClamp(soma(0.5))

In [ ]:
stim.delay = 200 * ms  # Wait 200ms before injecting, so we can see the resting state
stim.dur = 5000 * ms  # A LONG step: we need to tell sustained firing from a brief burst
stim.amp = initial_current * nA  # YOUR personal starting current, see Step 1b

# Name the start and end of the step, so we can ask for spikes "during the step"
# later on. NEURON gives these back as plain numbers in ms, hence the "* ms".
step_start = stim.delay * ms
step_end = (stim.delay + stim.dur) * ms
t_stop = step_end + 300 * ms  # run a little past the step, to see it switch off

## Step 3: Run the simulation

### Define recordings of simulation variables

In [ ]:
soma_v = h.Vector().record(soma(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

### Set the initial voltage

In [ ]:
h.finitialize( float(v_init) )

### Run the simulation

In [ ]:
h.continuerun( float(t_stop) )

## Step 4: Plot the results

In [ ]:
plt.plot(t, soma_v, lw=2, label="soma(0.5).v")
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("v [mV]", size=16)
plt.xticks(size=12)
plt.yticks(size=12)
#plt.axis([100,200,-80,30])

In [ ]:
def find_spikes(v, t, t_start=None, t_stop=None):
    """ Returns times of spikes for a voltage trace and time grid"""

    # look for upward crossing of 0mV
    v_arr = np.array(v)
    t_arr = np.array(t)
    # This is tricky & powerful notation! Let's discuss in class!
    spikes = t_arr[1:][(v_arr[1:]>0) & (v_arr[:-1]<0)]

    # Keep only the spikes inside the window asked for. We only ever want spikes
    # DURING the current step: the jump from v_init down to the resting potential
    # can fire one of its own before the stimulus has even started.
    if t_start is not None:
        spikes = spikes[spikes >= float(t_start)]
    if t_stop is not None:
        spikes = spikes[spikes <= float(t_stop)]
    return spikes

In [ ]:
spike_times = find_spikes(soma_v, t, step_start, step_end)

In [ ]:
spike_times, len(spike_times)

In [ ]:
firing_freq = (len(spike_times)/(stim.dur*ms)).rescale(Hz)

In [ ]:
firing_freq

In [ ]:
plt.plot(t, soma_v, lw=2, label="soma(0.5).v")
plt.plot(spike_times, len(spike_times)*[0], 'r.')
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("v [mV]", size=16)
plt.xticks(size=12)
plt.yticks(size=12)
# Zoom in on the start of the step, where the action is
plt.axis([190,260,-80,45])

### One spike, and then nothing?

There it is -- **a Hodgkin-Huxley action potential**. Note how far it overshoots:
it shoots up past 0 mV towards the sodium reversal potential, then undershoots the
resting potential on the way back down.

Now look at the whole trace again, or at `len(spike_times)`. Your cell fired
**once** and then went quiet, even though the current stays on for five full
seconds. That is not a bug, and it is not a badly chosen current: it is a real
property of this membrane, and the three questions at the end of the notebook are
about exactly that.

Two things are worth trying right now, by editing `stim.amp` in Step 2 and
re-running Steps 3 and 4:

- make the current a bit **smaller** and the spike disappears altogether;
- make it **two or three times bigger** and the cell fires continuously, for as
  long as you leave the current on.

So somewhere between those two settings the behaviour changes twice over. Finding
where is what the rest of this notebook does.

### Is it *still* firing at the end?

`firing_freq` above averages over the whole step, which is fine when the cell
fires steadily throughout. But we are about to hunt for the current at which
firing *begins*, and just below it the cell fires a short burst and then gives
up. To tell those two apart we need the firing rate at the **end** of the step.

In [ ]:
TAIL = 2000 * ms  # measure the sustained rate over the last 2s of the step

In [ ]:
def steady_firing_rate(spike_times, t_from, t_to):
    """ Firing rate at the end of the step in Hz, from the inter-spike intervals.

    Averaging the intervals (ISIs) inside a window, rather than counting spikes over
    the whole step: a count is quantised -- one spike more or less moves the answer
    by a few percent -- and it mixes in the settling at stimulus onset.

    Returns 0 Hz if fewer than two spikes land in the window, i.e. if the cell is not
    firing any more. That is exactly the test we need below.
    """
    tail = spike_times[(spike_times >= float(t_from)) & (spike_times <= float(t_to))]
    if len(tail) < 2:
        return 0 * Hz
    return (1 / (np.mean(np.diff(tail)) * ms)).rescale(Hz)

---

## Now it's your turn! Questions with answer feedback

Answer each question for **your** neuron -- the one you just simulated, using the
parameters printed in Step 1b. Each `submit` call grades one answer and leaves a record in StudiUM; you can re-run a cell to resubmit a better answer.

Answers are scored on relative error: within 5% earns full credit, and credit
fades to zero at 20% off. (Question 3 is scored more loosely -- see there.)

### The f-I curve

Everything below is built on one tool: a function that injects a given current,
runs the simulation and measures what came out. Sweeping it across a range of
currents gives the **f-I curve** -- this cell's input/output relationship, and
where the three graded answers are hiding.

In [ ]:
I_range = np.arange(0, 0.1, 0.004)  # nA. Every point costs a 5s simulation, so
                                    # don't ask for too many -- 25 shows the shape.

In [ ]:
def run_current(I_amp_nA):
    """ Inject a current, run the simulation, and measure the response.

    Returns (number of spikes during the step, firing rate at the END of the step).
    The second number is 0 Hz exactly when the cell is no longer firing -- which is
    how we tell sustained firing from a burst that dies out.
    """
    stim.amp = I_amp_nA * nA
    h.finitialize( float(v_init) )
    h.continuerun( float(t_stop) )

    spikes = find_spikes(soma_v, t, step_start, step_end)
    return len(spikes), steady_firing_rate(spikes, step_end - TAIL, step_end)

In [ ]:
def plot_trace(title="", t_from=None, t_to=None):
    """ Plot the voltage trace left behind by the most recent run_current() call.

    run_current re-uses the same recording vectors every time, so `soma_v` and `t`
    always hold whatever it simulated last -- handy, but it also means you have to
    re-run at the current you actually want to look at.
    """
    plt.figure(figsize=(10, 3))
    plt.plot(t, soma_v, lw=1.5)
    if t_from is not None:
        plt.xlim(float(t_from), float(t_to))
    plt.xlabel("t [ms]", size=14)
    plt.ylabel("v [mV]", size=14)
    plt.title(title, size=13)
    plt.show()

In [ ]:
# One simulation per current, so this takes a little while.
results = [run_current(I) for I in tqdm(I_range, desc="f-I curve")]
n_spikes = [n for n, rate in results]
rates = [float(rate) for n, rate in results]

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(7,7))
ax1.plot(I_range, n_spikes, 'x')
ax1.set_ylabel("spikes during the step", size=13)
ax2.plot(I_range, rates, 'x')
ax2.set_ylabel("rate at end of step [Hz]", size=13)
ax2.set_xlabel("injected current [nA]", size=14)

### What the f-I curve is telling us

Two things should jump out of those two panels:

1. There is a **threshold current** below which the cell never spikes at all.
2. Just above that threshold the cell fires **once**, or a brief burst, and then
   falls silent -- the top panel shows a spike or two while the bottom panel still
   reads 0 Hz. It only keeps firing for as long as the current is on once you pass
   a **second, higher** current. And when it finally does fire repetitively, it
   starts at around 40 Hz: there is **no way to make this cell fire slowly**.

That abrupt onset at a non-zero rate is what makes the Hodgkin-Huxley membrane a
**type II** excitable system. The three questions below are about those two
currents and that rate.

Your sweep locates the two currents only to within one sweep step, which is not
accurate enough. Bisection is: halve the bracket, keep whichever half still
contains the threshold, and repeat.

In [ ]:
def find_threshold(test, lo, hi, iters=18):
    """ Smallest current in [lo, hi] (nA) for which test(I) is True, by bisection.

    `test` takes a current in nA and returns True or False. Each pass halves the
    bracket, so after `iters` passes the answer is pinned down to (hi-lo)/2**iters.
    Needs test(lo) to be False and test(hi) to be True to get started.
    """
    assert not test(lo), "test is already True at lo -- lower the bracket"
    assert test(hi), "test is still False at hi -- raise the bracket"
    for _ in trange(iters, desc="bisecting", leave=False):
        mid = 0.5 * (lo + hi)
        if test(mid):
            hi = mid
        else:
            lo = mid
    return hi  # the end of the bracket we know satisfies the test

### Question 1 -- The rheobase

The **rheobase** is the smallest current step that makes the cell fire *at all* --
at least one spike. Find it for **your** neuron, in **nA**.

This first one is worked out for you, so you can see how `find_threshold` and
`run_current` fit together.

In [ ]:
rheobase = find_threshold(lambda I: run_current(I)[0] >= 1, 0.0, 0.1)

print(f"Submitting {rheobase:.5g} nA")
assignment.submit(rheobase, "rheobase")

Now look at what the cell does *right at* its threshold. Re-run there --
the bisection left the model sitting at its last trial current, not at the
answer -- and plot the first stretch of the step.

Try nudging the current a hair below `rheobase` and plotting again: the action
potential vanishes completely. There is no such thing as half a spike.

In [ ]:
n_spikes, rate = run_current(rheobase)
print(f"{n_spikes} spike(s) during the step, {float(rate):.1f} Hz at the end of it")

plot_trace(f"at the rheobase, {rheobase:.4g} nA",
           step_start - 20*ms, step_start + 100*ms)

### Question 2 -- The current needed for *sustained* firing

Just above the rheobase your cell fires a spike or two and then stops, no matter
how long you leave the current on. Find the smallest current at which it is still
firing at the **end of the 5 s step**, in **nA**.

`run_current` hands you what you need: its second return value is the firing rate
at the end of the step, and that is `0 Hz` precisely when the cell has given up.
Write that test and pass it to `find_threshold`.

In [ ]:
# Fill in the test below: is the cell still firing at the end of the step?
i_sustained = find_threshold(lambda I: None, 0.0, 0.1)  # <-- replace None

print(f"Submitting {i_sustained:.5g} nA")
assignment.submit(i_sustained, "i_sustained")

And now the other side of it. Plot the same first second of the step at
this second threshold, and put it next to the single spike from Question 1.

The current has gone up by well under a factor of three, and the cell has stopped
firing once and started firing indefinitely. Notice too how *fast* the train is
the moment it appears -- count the spikes in the plot and you have most of the
answer to Question 3 already.

In [ ]:
n_spikes, rate = run_current(i_sustained)
print(f"{n_spikes} spikes during the step -- and still going when it ends")

plot_trace(f"at the sustained-firing threshold, {i_sustained:.4g} nA",
           step_start - 20*ms, step_start + 1000*ms)

### Question 3 -- The minimum sustained firing rate

At the current you just found, how fast does your cell fire? This is the
**slowest rate it can sustain**: with any less current it does not fire
repetitively at all. Report it in **Hz**.

Worth comparing against the rate at, say, twice that current, and against the
firing rates you know from real recordings. A cortical neuron ticking along at
5 Hz is doing something this model simply cannot do.

> **A note on scoring.** This answer is graded more loosely -- full credit within
> 10%, fading to zero at 25% off. The rate climbs very steeply the moment you
> step past the threshold current (roughly as the *square root* of how far past it
> you are), so the number you get depends on how tightly you pinned down
> Question 2. The physics is what is being marked: tens of Hz, not a few Hz.

In [ ]:
min_firing_rate = None  # <-- replace with your answer, in Hz

assignment.submit(min_firing_rate, "min_firing_rate")